In [1]:
import pathlib
import re
# import argparse

# import yaml
# import pandas as pd
import numpy as np
import scipy
from sklearn.model_selection import train_test_split
import pandas as pd

import jax
import jax.numpy as jnp
import jax.random as jr
import optax
import equinox as eqx
from jaxtyping import Array, Float, Int, PyTree, PRNGKeyArray

# from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader, Subset

import altair as alt

import trapallada.data
import trapallada.model.signal

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# input_directory = pathlib.Path.cwd().parent / "light curves" / "Simulations"
input_directory = pathlib.Path.cwd().parent / "light curves"
assert input_directory.exists()

In [4]:
base_filename = 'small'

signals_file = input_directory / f'{base_filename}_signals.pt'
assert signals_file.exists()

metadata_file = input_directory / f'{base_filename}_labels.parquet'
assert metadata_file.exists

In [5]:
ds = trapallada.data.SignalsDataset(signals_file, metadata_file)
len(ds)

18000

In [6]:
ds.metadata_df.head()

,label,std,file name
0,Constant,1.0,Constant/cLC_s1_793.jpg
1,Constant,3.0,Constant/cLC_s3_524.jpg
2,Constant,3.0,Constant/cLC_s3_956.jpg
3,Constant,3.0,Constant/cLC_s3_283.jpg
4,Constant,5.0,Constant/cLC_s5_666.jpg


In [7]:
i = ds.metadata_df[ds.metadata_df['label'] == 'AM'].index[0]
ds.metadata_df.loc[i]

label                    AM
std                     2.0
file name    AM/AM_s2_3.jpg
Name: 12000, dtype: object

In [8]:
y = ds[i][1]
df = pd.DataFrame(y.squeeze(), columns=['signal'])
df


,signal
0,2.167470
1,2.730022
2,2.250210
3,2.036631
4,2.202767
...,...
495,2.698509
496,3.435666
497,4.214709
498,2.854706


In [9]:
alt.Chart(df.reset_index()).mark_line().encode(
    x='index', y='signal'
)

alt.Chart(...)

In [10]:
rng_key = jr.PRNGKey(42)

parameters = {
    'training': {
        'splitting': {
            'test_size':  0.2,
            'random_state': 42
        },
        'batch_size': 32,
        'learning_rate': 1e-3,
    },
}

In [11]:
train_idx, validation_idx = train_test_split(
    np.arange(len(ds)), 
    **parameters['training']['splitting'], 
    shuffle=True,
    stratify=ds.labels
)
train_ds, validation_ds = Subset(ds, train_idx), Subset(ds, validation_idx)

# a random key...
key, subkey = jr.split(rng_key)

# ...to compute a seed for a PyTorch `Generator`...
torch_seed = jr.bits(subkey, (1,), dtype=jnp.uint32).item()
torch_rng = torch.Generator().manual_seed(torch_seed)

# ...meant for "batching" in the `DataLoader`'s
train_dl = DataLoader(train_ds, parameters['training']['batch_size'], shuffle=True, generator=torch_rng)
validation_dl = DataLoader(validation_ds, batch_size=parameters['training']['batch_size'], shuffle=False, generator=torch_rng)

The indexes (1st element in the `tuple` for a batch) are used to build a `DataFrame`

In [12]:
batch = next(iter(train_dl))
batch_metadata_df = ds.metadata_df.loc[batch[0]]
df = pd.DataFrame(batch[1].squeeze().T, columns=batch_metadata_df['file name'].str.extract(r'.*/(.*).jpg')[0])
df

,AM_s1_10,AM_s4_785,cLC_s2_700,cLC_s3_227,cLC_s1_239,FM_s2_405,AM_s1_14,cLC_s10_767,cLC_s5_567,cLC_s2_534,...,AM_s2_924,cLC_s10_821,AM_s4_318,cLC_s10_730,AM_s4_292,FM_s3_150,FM_s1_298,FM_s10_790,cLC_s5_247,cLC_s2_262
0,26.004570,18.392358,-1.011089,-1.928161,-0.329188,10.137430,6.014218,2.718772,-7.667392,-0.093223,...,27.269895,4.104799,7.351490,13.171178,38.050363,41.287650,0.650699,3.349828,0.184828,2.003481
1,-1.137189,11.367001,1.985901,-3.306581,3.227127,-3.691805,3.719498,64.371705,12.801018,1.214442,...,1.040360,32.226270,11.993502,7.848760,23.076989,35.379201,0.810965,5.998908,8.790745,14.450249
2,-25.151938,12.467173,4.328978,-1.072125,3.167349,-12.202347,5.895899,-27.468931,15.386414,-1.414108,...,-21.763413,-5.127012,11.288275,46.703287,2.277472,27.998921,1.385281,22.289323,-4.705386,23.655461
3,7.947718,1.785836,4.803749,1.299866,5.336854,8.482534,3.353507,-27.969041,1.999349,5.109497,...,-11.912160,-13.781504,6.025080,36.350750,-19.125833,26.682685,1.302174,-14.553110,7.168589,27.454692
4,23.406499,-8.846379,4.666235,0.670014,6.355245,9.820031,2.940245,29.833602,-8.144506,4.942709,...,22.078796,14.470655,2.678741,12.276250,-31.888764,18.843212,0.977276,-2.846855,10.233791,23.968761
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,-20.019297,1.047639,-11.962867,-2.146186,16.151688,-12.035517,-1.118431,-44.792471,-6.967641,-9.405355,...,-0.963410,31.207377,-3.473874,11.960862,30.127132,-32.761021,0.343630,-5.476005,10.886471,23.026306
496,-12.967754,3.171007,-14.272316,-2.348273,13.729345,5.126830,-2.099006,11.726413,-3.935882,-7.599225,...,-2.219942,18.203955,-7.023217,-13.107886,28.561946,-33.801316,1.044479,-0.200786,1.121516,19.883161
497,22.786259,0.275734,-14.909333,-5.738895,12.985340,9.340226,-3.189995,27.089411,5.893922,-9.266805,...,0.602317,-15.484464,1.008754,-41.699821,10.474119,-34.959640,2.661667,-17.831164,1.959869,8.378678
498,5.142911,-1.129565,-15.384109,-9.188903,11.398479,-2.691309,-2.701782,-57.458888,14.319077,-6.900059,...,0.917965,-18.348155,6.063433,-25.482979,-6.513850,-29.459700,0.814356,-10.205461,-2.041880,-4.534463


In [13]:
alt.Chart(
    df.reset_index()
).transform_fold(
    df.columns.to_list()
).mark_line().encode(
    x='index:Q', y='value:Q', color='key:N'
).properties(width=1_200, height=400)

alt.Chart(...)

In [14]:
signals_len = ds[0][1].shape[-1]
signals_len

500

In [15]:
print(isinstance(eqx.nn.Linear(1, 2, key=jr.PRNGKey(42)), eqx.nn.Dropout))
print(isinstance(eqx.nn.Dropout(0.2), eqx.nn.Dropout))

False
True


In [16]:
x = batch[1][0].numpy()
model = trapallada.model.signal.Classifier(
    input_size=signals_len, kernel_sizes=[128, 64, 32], channels=[1, 12, 12, 12], stride=2, dropout_rate=0.25,
    key=jr.PRNGKey(42))
model.debug_call(x, jr.PRNGKey(42))
print(model)

input_size=(187,)
input_size=(62,)
input_size=(16,)
192
after equinox.nn._conv: (12, 187)
after jax._src.nn.functions: (12, 187)
after equinox.nn._normalisation: (12, 187)
after equinox.nn._dropout: (12, 187)
after equinox.nn._conv: (12, 62)
after jax._src.nn.functions: (12, 62)
after equinox.nn._normalisation: (12, 62)
after equinox.nn._dropout: (12, 62)
after equinox.nn._conv: (12, 16)
after jax._src.nn.functions: (12, 16)
after equinox.nn._normalisation: (12, 16)
after equinox.nn._dropout: (12, 16)
after jax._src.numpy.lax_numpy: (192,)
after equinox.nn._linear: (3,)
Classifier(
  layers=[
    Conv1d(
      num_spatial_dims=1,
      weight=f32[12,1,128],
      bias=f32[12,1],
      in_channels=1,
      out_channels=12,
      kernel_size=(128,),
      stride=(2,),
      padding=((0, 0),),
      dilation=(1,),
      groups=1,
      use_bias=True
    ),
    <wrapped function relu>,
    LayerNorm(
      shape=(12, 187),
      eps=1e-05,
      use_weight=True,
      use_bias=True,
      

In [17]:
# jax.vmap(model)(x, jr.split(jr.PRNGKey(42), len(x)))

In [18]:
def loss(model: eqx.Module, x: Float[Array, 'batch length'], y: Float[Array, 'batch'], key: PRNGKeyArray):

    y_pred = jax.vmap(model)(x, jr.split(key, len(x)))
    
    return optax.losses.softmax_cross_entropy_with_integer_labels(y_pred, y).mean()

In [19]:
@eqx.filter_jit
def take_step(
    model,
    opt_state: PyTree,
    x: Float[Array, 'batch 1 length'],
    y: Float[Array, 'batch'],
    key: PRNGKeyArray
) -> tuple[tuple[eqx.Module], PyTree, Float[Array, '']]:

    value, grads = eqx.filter_value_and_grad(loss)(model, x, y, key)

    # it returns transformed/processed gradients, and the optimizer state
    updates, opt_state = optim.update(grads, opt_state, model)

    model = eqx.apply_updates(model, updates)

    return model, opt_state, value

In [20]:
def make_results_df(n_epochs: int) -> pd.DataFrame:
    
    return pd.DataFrame({ (data, metric): [0.] * n_epochs for data in ['training', 'validation'] for metric in ['loss', 'accuracy']})

make_results_df(2)

training          validation         
      loss accuracy       loss accuracy
0      0.0      0.0        0.0      0.0
1      0.0      0.0        0.0      0.0

In [21]:
@eqx.filter_jit
def compute_accuracy(model: eqx.Module, x: Float[Array, "batch 1 500"], y: Float[Array, "batch"], key: PRNGKeyArray) -> Float[Array, ""]:
    
    y_pred = jnp.argmax(jax.nn.log_softmax(jax.vmap(model)(x, jr.split(key, len(x)))), axis=-1)
    
    return (y_pred == y).mean()

In [22]:
# _, x, y = next(iter(train_dl))
# compute_accuracy(model, x.numpy(), y.numpy(), jr.PRNGKey(42))

In [23]:
def evaluate(model: eqx.Module, dataloader: torch.utils.data.DataLoader):

    # a fixed key is used since we are not training
    key = jr.PRNGKey(42)

    # in case, there are layers that behave differently in training and inference modes
    model = eqx.nn.inference_mode(model)

    loss_acc = 0.
    accuracy_acc = 0.

    for _, x, y in dataloader:

        x = x.numpy()
        y = y.numpy()
    
        loss_value = loss(model, x, y, key)
        
        # adding up *averages* (of both the loss and the accuracy)
        loss_acc += loss_value.item()
        accuracy_acc += compute_accuracy(model, x, y, key).item()

    # reverting to training mode
    model = eqx.nn.inference_mode(model, value=False)

    return loss_acc / len(dataloader), accuracy_acc / len(dataloader)


In [24]:
parameters['model'] = {
    'kernel_sizes': [128, 64, 32],
    # 'channels': [1, 12, 12, 12],
    'channels': [1, 8, 8, 8],
    'stride': 2,
    'dropout_rate': 0.25,
}

In [25]:
# # channels = [1, 4, 4, 4]
# channels = [1, 8, 8, 8]
# # channels = [1, 12, 12, 12]
model = trapallada.model.signal.Classifier(
    input_size=signals_len,
    kernel_sizes=parameters['model']['kernel_sizes'],
    channels=parameters['model']['channels'],
    stride=parameters['model']['stride'],
    dropout_rate=parameters['model']['dropout_rate'],
    key=jr.PRNGKey(42))
optim = optax.adamw(parameters['training']['learning_rate'])
opt_state = optim.init(eqx.filter(model, eqx.is_array))

input_size=(187,)
input_size=(62,)
input_size=(16,)
128


In [26]:
# n_epochs = 100
parameters['training']['n_epochs'] = 100

results_df = make_results_df(parameters['training']['n_epochs'])

for epoch in range(parameters['training']['n_epochs']):

    for _, x, y in train_dl:

        key, key2, key3 = jr.split(key, 3)

        x = x.numpy()
        y = y.numpy()

        model, opt_state, loss_value = take_step(model, opt_state, x, y, key2)

        # adding up *averages* (of both the loss and the accuracy)
        results_df.loc[epoch, ('training', 'loss')] += loss_value.item()
        results_df.loc[epoch, ('training', 'accuracy')] += compute_accuracy(model, x, y, key3)


    results_df.loc[epoch, [('training', 'loss'), ('training', 'accuracy')]] /= len(train_dl)

    # validation
    results_df.loc[epoch, [('validation', 'loss'), ('validation', 'accuracy')]] = evaluate(model, validation_dl)

    print(
        epoch,
        results_df.loc[epoch, ('training', 'loss')],
        results_df.loc[epoch, ('validation', 'loss')],
        '|||',
        results_df.loc[epoch, ('training', 'accuracy')],
        results_df.loc[epoch, ('validation', 'accuracy')],
    )

0 0.749416379597452 0.5952866488853387 ||| 0.6286111111111111 0.7253871681415929
1 0.596553320619795 0.5042462159047084 ||| 0.7329166666666667 0.7925884955752213
2 0.49891422473722036 0.40423444731045616 ||| 0.8004166666666667 0.8451327433628318
3 0.4142261161737972 0.3180595590873102 ||| 0.8365277777777778 0.8725110619469026
4 0.3694019707706239 0.3171931228938356 ||| 0.8568055555555556 0.8689159292035398
5 0.334646898176935 0.2688499013934515 ||| 0.8677083333333333 0.8841261061946902
6 0.3158121325406763 0.2797665054028013 ||| 0.8756944444444444 0.8915929203539823
7 0.2969332551293903 0.2505122824052794 ||| 0.8808333333333334 0.9029314159292036
8 0.2842090977066093 0.2519114808698671 ||| 0.8869444444444444 0.900995575221239
9 0.27492142650816176 0.23739264884024067 ||| 0.8917361111111111 0.9073561946902655
10 0.2713753830931253 0.22310169034563335 ||| 0.8959722222222222 0.9128871681415929
11 0.2513384096490012 0.21137614584470218 ||| 0.900625 0.9192477876106194
12 0.25123200201325946

In [27]:
training_df = results_df.loc[:epoch-1,[('training', 'loss'), ('validation', 'loss')]]
training_df.columns = ["_".join(a) for a in training_df.columns.to_flat_index()]
training_df.tail()
alt.Chart(
    training_df.reset_index()
).transform_fold(
    training_df.columns.to_list()
).mark_line().encode(
    x='index:Q', y='value:Q', color='key:N'
).properties(width=1_200, height=400)

alt.Chart(...)